# Driving complaints, collisions, and stops by police beat

## Preparing the 2024–2025 comparison

Count distinct vehicle stops and collision events using a common local-date window and a fixed City of Berkeley beat map. Complaint counts and rank correlations await the shared calls data.

**A5.** Do beats with more reckless-driving complaints also have more collisions and vehicle stops?

**Inputs**  
`data/boundaries/police_beats.geojson`, `data/processed/vehicle_stops.csv`, and `data/processed/collisions.csv`.

**Output**  
`data/processed/a5_driving_by_beat.csv` contains one row per beat. The exclusion table below separately reports undated records from the full snapshots and spatial exclusions among dated 2024–2025 events.

In [1]:
import json
from bpdtools.paths import repo_path

import numpy as np
import pandas as pd
from matplotlib.path import Path as PolygonPath

data = repo_path("data")
features = json.loads((data / "boundaries/police_beats.geojson").read_text())["features"]
stops = pd.read_csv(data / "processed/vehicle_stops.csv")
collisions = pd.read_csv(data / "processed/collisions.csv")

In [2]:
def assign_beats(points, features):
    """Assign points to beats using the supplied boundary polygons.

    Parameters
    ----------
    points : numpy.ndarray
        N-by-2 array of longitude and latitude in degrees.
    features : list of dict
        GeoJSON Polygon or MultiPolygon features with a Sector property.

    Returns
    -------
    pandas.Series
        One beat ID or exclusion reason per input point, in input order.

    Raises
    ------
    ValueError
        If a feature has an unsupported geometry type.
    """
    valid = np.isfinite(points).all(axis=1)
    assignments = np.full(len(points), "outside", dtype=object)
    assignments[~valid] = "missing"
    for feature in features:
        beat = str(feature["properties"]["Sector"])
        geometry = feature["geometry"]
        parts = geometry["coordinates"]
        if geometry["type"] == "Polygon":
            parts = [parts]
        elif geometry["type"] != "MultiPolygon":
            raise ValueError("Expected polygon geometry")
        inside = np.zeros(len(points), dtype=bool)
        boundary = np.zeros(len(points), dtype=bool)
        for shell, *holes in parts:
            part = np.ones(len(points), dtype=bool)
            for index, ring in enumerate([shell, *holes]):
                path = PolygonPath(ring)
                # Opposite tiny radii flag exact edge points regardless of winding.
                a = path.contains_points(points, radius=1e-10)
                b = path.contains_points(points, radius=-1e-10)
                boundary |= a ^ b
                selected = a & b
                part &= selected if index == 0 else ~selected
            inside |= part
        # Exclude points on polygon edges or inside multiple beat interiors.
        ambiguous = boundary | (inside & (assignments != "outside"))
        assignments[inside & valid] = beat
        assignments[ambiguous & valid] = "ambiguous"
    return pd.Series(assignments)


def in_window(frame, column, fmt):
    """Select records in the shared two-year window and count undated records.

    Parameters
    ----------
    frame : pandas.DataFrame
        Full event snapshot.
    column : str
        Column containing the local event date.
    fmt : str
        Expected date format for nonmissing values.

    Returns
    -------
    selected : pandas.DataFrame
        Records with a known local date in 2024 or 2025.
    missing_dates : int
        Records in the full snapshot whose year cannot be determined.
    """
    dates = pd.to_datetime(frame[column], format=fmt)
    # Apply the same calendar-year filter to both sources and audit missing dates.
    return frame.loc[dates.dt.year.isin([2024, 2025])], int(dates.isna().sum())


stops, undated_stops = in_window(stops, "StopDate", "%m/%d/%Y")
collisions, undated_collisions = in_window(collisions, "Date", "%Y-%m-%d")
stop_points = stops[["Longitude", "Latitude"]].to_numpy()
# Convert spherical Web Mercator metres to longitude and latitude in degrees.
radius_m = 6378137
lon = np.degrees(collisions["x"] / radius_m)
lat = np.degrees(np.arctan(np.sinh(collisions["y"] / radius_m)))
counts = pd.DataFrame({
    "vehicle_stops": assign_beats(stop_points, features).value_counts(),
    "collisions": assign_beats(np.column_stack([lon, lat]), features).value_counts(),
}).fillna(0).astype(int)
audit = counts.reindex(["missing", "outside", "ambiguous"], fill_value=0).rename_axis("exclusion")
audit.loc["missing local date (full snapshot)"] = [undated_stops, undated_collisions]
display(audit)

,vehicle_stops,collisions
exclusion,,
missing,41,0
outside,365,39
ambiguous,0,3
missing local date (full snapshot),0,101


In [3]:
beats = sorted({str(f["properties"]["Sector"]) for f in features}, key=int)
table = counts.reindex(beats, fill_value=0).rename_axis("beat")
excluded = counts.loc[~counts.index.isin(beats)].sum()
# Reconcile assigned and spatially excluded events within the dated window.
assert (table.sum() + excluded).tolist() == [len(stops), len(collisions)]
table.to_csv(data / "processed/a5_driving_by_beat.csv")
display(table)

,vehicle_stops,collisions
beat,,
1,73,128
2,340,149
3,814,122
4,618,76
5,560,149
6,548,159
7,419,145
8,221,94
9,458,130
